# LC13 — A language model you can read (self-paced, ~45 min)

Before Period 2 puts an AI coding agent next to you, we build the smallest thing that deserves the name *language model*: a *bigram model*. It fits in one screen of Python, you can inspect every number in it, and it already has the one control knob (temperature) you will keep using in the real thing. By the end you will know exactly what a language model does — and exactly why the real ones need to be so much bigger.

No mathematics beyond counting and dividing. No machine-learning library. Just Python.

In [1]:
# Install exactly what this notebook uses.
%pip install numpy --quiet
import re
import numpy as np
from collections import Counter, defaultdict
from pathlib import Path


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /private/tmp/claude-501/-Users-larsnordstrom-code-EG2140-dev/b312b5c1-fbe4-4a5d-996d-64114a3bca9a/scratchpad/lc01venv/bin/python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the training text lives one level up in ../labs and
# ../guides. Failing here, early and clearly, beats a confusing error later.
assert Path("../labs").exists() and Path("../guides").exists(), (
    "Course folders not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

## 1. A corpus you have already read

A language model learns from text — a *corpus*. Ours: the course's own lab instructions and guides. That choice is deliberate: you know this text well, so you can judge the model's output the way you judged power-flow numbers in Period 1 — against knowledge you already have.

In [3]:
# Read every lab and guide, dropping the fenced code blocks (```...```) —
# we want the model to learn prose, not pip commands.
texts = []
for f in sorted(Path("../labs").glob("*.md")) + sorted(Path("../guides").glob("*.md")):
    raw = f.read_text(encoding="utf-8")
    prose = re.sub(r"```.*?```", " ", raw, flags=re.DOTALL)   # remove code fences
    texts.append(prose)
corpus_text = " ".join(texts)

# Tokenise: lowercase words only. Real models use smarter units ("tokens"),
# but words keep everything readable today.
tokens = re.findall(r"[a-zåäö]+", corpus_text.lower())
print(f"{len(tokens)} words of training text, {len(set(tokens))} distinct")
print("first twelve:", tokens[:12])

7260 words of training text, 1469 distinct
first twelve: ['lab', 'start', 'the', 'svedala', 'toolbox', 'from', 'notebook', 'to', 'package', 'eg', 'in', 'pairs']


## 2. Training = counting

A bigram model answers one question: *given the current word, what word comes next?* Training it means walking through the corpus once and counting every pair of neighbours. That is the entire "learning".

In [4]:
# For each word, count what follows it. This loop IS the training:
# one pass over the corpus, nothing but bookkeeping.
follows = defaultdict(Counter)
for w, nxt in zip(tokens, tokens[1:]):
    follows[w][nxt] += 1

# Inspect the model like any other table: what does it know about "power"?
print('After "power" the corpus continues with:')
for word, count in follows["power"].most_common(5):
    print(f"   {word:12s} seen {count} times")

After "power" the corpus continues with:
   flow         seen 7 times
   flows        seen 2 times
   system       seen 1 times


## 3. From counts to probabilities

Counts become a probability distribution by dividing each by the row's total. That table of rows — one distribution per word — **is the whole model**. Every "prediction" is just a lookup.

In [5]:
def next_distribution(word):
    """The model's entire knowledge about one word: P(next | word)."""
    counts = follows[word]
    total = sum(counts.values())
    words = list(counts.keys())
    # Divide every count by the row total -> probabilities that sum to 1.
    probs = np.array([counts[w] for w in words], dtype=float) / total
    return words, probs

words, probs = next_distribution("the")
# Show the five most likely continuations of "the".
for w, p in sorted(zip(words, probs), key=lambda t: -t[1])[:5]:
    print(f'P({w!r} | "the") = {p:.3f}')

P('other' | "the") = 0.030
P('course' | "the") = 0.023
P('same' | "the") = 0.023
P('week' | "the") = 0.019
P('bug' | "the") = 0.017


## 4. Generation = sampling

To generate text, start from a word, draw the next one from its distribution, move there, repeat. Note what this means: the model never plans a sentence — each word is drawn looking at **one** word of history.

In [6]:
rng = np.random.default_rng(0)   # fixed seed: same "random" text every run

def generate(start, n_words=25, temperature=1.0):
    """Sample a chain of words, one bigram step at a time."""
    out = [start]
    # Each pass: look up the current word's distribution, draw one successor.
    for _ in range(n_words):
        words, probs = next_distribution(out[-1])
        # Temperature reshapes the distribution before drawing (section 5).
        p = probs ** (1.0 / temperature)
        p = p / p.sum()
        out.append(rng.choice(words, p=p))
    return " ".join(out)

print(generate("the"))

the choice and prints them a requirements txt recipe never be cryptic part b what they are true python macos linux and the deliverable the fetch


Read that aloud. Every consecutive *pair* of words occurs somewhere in the course material — locally it sounds right — yet the sentence as a whole goes nowhere. Keep that observation; it becomes the punchline of section 6.

## 5. The temperature knob

Temperature rescales the probabilities before sampling: each probability is raised to the power $1/T$ and the row is re-normalised. Low $T$ exaggerates the differences — the most common continuation wins almost always. High $T$ flattens them — rare continuations get their chance. You will meet this exact parameter again in every AI API you touch in Period 2.

In [7]:
# Same start word, same model — only the knob moves.
for T in (0.3, 1.0, 2.0):
    print(f"T={T}:")
    print("  ", generate("your", n_words=20, temperature=T))
    print()

T=0.3:
   your lab add at the other pair s repo lab partner during the test network is the same tests test that

T=1.0:
   your scratch folder shows one commit with your github username sent after the folder and docstrings kept honest checkpoint git switch

T=2.0:
   your challenger method mae skill percentage if name eq bus zone plot it catches the end anything works for b the



Typical result: at `T=0.3` the model plays it safe and loops through the corpus's most-worn phrases; at `T=2.0` it free-associates into word salad; `T=1.0` sits between. Neither extreme is "wrong" — it is a dial you set per task. (Asking an agent for reproducible code edits and asking it to brainstorm test ideas want different settings.)

## 6. Why the real ones are so much bigger

Diagnosis time. Our model's failure is not the sampling — it is the **one word of memory**. Let's measure how little it takes before the model has, in effect, memorised the corpus instead of understanding it.

In [8]:
# How many choices does the model actually have, on average?
# For each word: how many distinct successors did the corpus show it?
branching = [len(c) for c in follows.values()]
print(f"average continuations per word: {np.mean(branching):.1f}")
print(f"words with only ONE continuation: "
      f"{sum(1 for b in branching if b == 1)} of {len(branching)}")

# Check a generated pair-chain against the corpus: every adjacent pair the
# model produces was literally seen in training.
sample = generate("the", n_words=15).split()
pairs_in_corpus = sum((b in follows[a]) for a, b in zip(sample, sample[1:]))
print(f"generated pairs found verbatim in the corpus: "
      f"{pairs_in_corpus} of {len(sample)-1}")

average continuations per word: 3.7
words with only ONE continuation: 724 of 1469
generated pairs found verbatim in the corpus: 15 of 15


Every pair it produces is a quotation; only the *chaining* is new. To do better, a model must look further back — and that is where counting collapses. With our vocabulary, a table over two-word histories would need millions of rows, three words a billion, and almost all of them would have **zero observations**. Counting cannot generalise to histories it never saw.

Modern language models break out of this with three moves you now have the vocabulary for:

- **Longer context, without a table** — instead of looking up the history, they *compute* with it: today's models weigh hundreds of thousands of tokens of history when predicting the next one (our model: one).
- **Meaning as numbers** — words become vectors, so "transformer" and "trafo" can share evidence instead of being unrelated rows. Unseen histories stop being dead ends.
- **Learned weights instead of counts** — billions of parameters tuned by gradient descent on trillions of tokens, so the "table" is replaced by a function that generalises.

But the loop you wrote today — *look at the context, produce a distribution over the next token, sample from it (with temperature), append, repeat* — is **exactly** the loop running inside the agents you will work with in Period 2. They are not a different kind of thing. They are this thing, scaled, with all the strengths and failure modes that follow: fluent locally, unanchored globally, tunable by the same knob you just turned. When an agent's confident code turns out subtly wrong in Lab 4 fashion — remember where the words come from.